In [43]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [10]:
lr = LinearRegression()


ames = pd.read_csv("AmesHousing.csv")
X = ames[["Gr Liv Area", "TotRms AbvGrd"]]
y = ames["SalePrice"]



X_train, X_test, y_train, y_test = train_test_split(X, y)

X_train_s = (X_train - X_train.mean())/X_train.std()

lr_fitted = lr.fit(X_train_s, y_train)
lr_fitted.coef_

array([ 74148.54085717, -19734.8040904 ])

In [11]:
X_test_s = (X_test - X_train.mean())/X_train.std()
y_preds = lr_fitted.predict(X_test_s)

r2_score(y_test, y_preds)

0.4641043143704079

In [12]:
new_house_s = (new_house - X_train.mean())/X_train.std()
lr_fitted.predict(new_house_s)

array([97554.96050742])

Creating a Pipeline Objects

In [21]:
lr = LinearRegression()

lr_pipeline = Pipeline(
  [StandardScaler(),lr]
)

lr_pipeline

TypeError: 'LinearRegression' object is not subscriptable

In [14]:
lr_pipeline = Pipeline(
  [("standardize", StandardScaler()),
  ("linear_regression", LinearRegression())]
)

lr_pipeline

Pipeline(steps=[('standardize', StandardScaler()),
                ('linear_regression', LinearRegression())])

In [15]:
lr_pipeline_fitted = lr_pipeline.fit(X_train, y_train)

y_preds = lr_pipeline_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.4641043143704079

In [16]:
lr_pipeline_fitted.predict(new_house)

array([97554.96050742])

In [24]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)


lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
)

lr_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('dummify',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['Bldg Type']),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area',
                                                   'TotRms AbvGrd'])])),
                ('linear_regression', LinearRegression())])

In [25]:
X = ames.drop("SalePrice", axis = 1)
y = ames["SalePrice"]



X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_fitted = lr_pipeline.fit(X_train, y_train)

In [26]:
ct_fitted = ct.fit(X_train)

ct.transform(X_train)

array([[ 0.        ,  0.        ,  0.        , ...,  1.        , -0.08398618,  0.360294  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.40352834, -0.2801315 ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.19361608,  0.360294  ],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  2.14881472,  2.2815705 ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.00609777,  0.360294  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.43926412,  0.360294  ]])

In [27]:
ct.transform(X_test)

array([[ 1.        ,  0.        ,  0.        , ...,  0.        , -0.6032422 , -0.920557  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.51315825,  0.360294  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.11194612,  0.360294  ],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.43947684, -0.920557  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  1.52770463,  1.641145  ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.95452587,  0.360294  ]])

In [28]:
lr_pipeline_fitted.named_steps['linear_regression'].coef_

array([ 74131.6639841 , -19730.31227195])

In [29]:
lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")


ct.fit_transform(X_train)

,dummify__Bldg Type_1Fam,dummify__Bldg Type_2fmCon,dummify__Bldg Type_Duplex,dummify__Bldg Type_Twnhs,dummify__Bldg Type_TwnhsE,standardize__Gr Liv Area,standardize__TotRms AbvGrd
1047,0.0,0.0,0.0,0.0,1.0,-0.083986,0.360294
1975,1.0,0.0,0.0,0.0,0.0,-0.403528,-0.280131
2601,1.0,0.0,0.0,0.0,0.0,0.193616,0.360294
1133,1.0,0.0,0.0,0.0,0.0,0.329422,1.000719
1845,0.0,0.0,0.0,1.0,0.0,0.149679,-1.560982
...,...,...,...,...,...,...,...
1569,1.0,0.0,0.0,0.0,0.0,0.381347,-0.280131
2564,1.0,0.0,0.0,0.0,0.0,-1.122498,-0.920557
2103,1.0,0.0,0.0,0.0,0.0,2.148815,2.281570
1731,1.0,0.0,0.0,0.0,0.0,-0.006098,0.360294


In [30]:
ct_inter = ColumnTransformer(
  [
    ("interaction", PolynomialFeatures(interaction_only = True), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
).set_output(transform = "pandas")

ct_inter.fit_transform(X_train)

,interaction__1,interaction__Gr Liv Area,interaction__TotRms AbvGrd,interaction__Gr Liv Area TotRms AbvGrd
1047,1.0,1456.0,7.0,10192.0
1975,1.0,1296.0,6.0,7776.0
2601,1.0,1595.0,7.0,11165.0
1133,1.0,1663.0,8.0,13304.0
1845,1.0,1573.0,4.0,6292.0
...,...,...,...,...
1569,1.0,1689.0,6.0,10134.0
2564,1.0,936.0,5.0,4680.0
2103,1.0,2574.0,10.0,25740.0
1731,1.0,1495.0,7.0,10465.0


In [31]:
ct_dummies = ColumnTransformer(
  [("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"])],
  remainder = "passthrough"
).set_output(transform = "pandas")

ct_inter = ColumnTransformer(
  [
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__TotRms AbvGrd", "dummify__Bldg Type_1Fam"]),
  ],
  remainder = "drop"
).set_output(transform = "pandas")

X_train_dummified = ct_dummies.fit_transform(X_train)
X_train_dummified

,dummify__Bldg Type_1Fam,dummify__Bldg Type_2fmCon,dummify__Bldg Type_Duplex,dummify__Bldg Type_Twnhs,dummify__Bldg Type_TwnhsE,remainder__Order,remainder__PID,remainder__MS SubClass,remainder__MS Zoning,remainder__Lot Frontage,remainder__Lot Area,remainder__Street,remainder__Alley,remainder__Lot Shape,remainder__Land Contour,remainder__Utilities,remainder__Lot Config,remainder__Land Slope,remainder__Neighborhood,remainder__Condition 1,remainder__Condition 2,remainder__House Style,remainder__Overall Qual,remainder__Overall Cond,remainder__Year Built,remainder__Year Remod/Add,remainder__Roof Style,remainder__Roof Matl,remainder__Exterior 1st,remainder__Exterior 2nd,remainder__Mas Vnr Type,remainder__Mas Vnr Area,remainder__Exter Qual,remainder__Exter Cond,remainder__Foundation,remainder__Bsmt Qual,remainder__Bsmt Cond,remainder__Bsmt Exposure,remainder__BsmtFin Type 1,remainder__BsmtFin SF 1,remainder__BsmtFin Type 2,remainder__BsmtFin SF 2,remainder__Bsmt Unf SF,remainder__Total Bsmt SF,remainder__Heating,remainder__Heating QC,remainder__Central Air,remainder__Electrical,remainder__1st Flr SF,remainder__2nd Flr SF,remainder__Low Qual Fin SF,remainder__Gr Liv Area,remainder__Bsmt Full Bath,remainder__Bsmt Half Bath,remainder__Full Bath,remainder__Half Bath,remainder__Bedroom AbvGr,remainder__Kitchen AbvGr,remainder__Kitchen Qual,remainder__TotRms AbvGrd,remainder__Functional,remainder__Fireplaces,remainder__Fireplace Qu,remainder__Garage Type,remainder__Garage Yr Blt,remainder__Garage Finish,remainder__Garage Cars,remainder__Garage Area,remainder__Garage Qual,remainder__Garage Cond,remainder__Paved Drive,remainder__Wood Deck SF,remainder__Open Porch SF,remainder__Enclosed Porch,remainder__3Ssn Porch,remainder__Screen Porch,remainder__Pool Area,remainder__Pool QC,remainder__Fence,remainder__Misc Feature,remainder__Misc Val,remainder__Mo Sold,remainder__Yr Sold,remainder__Sale Type,remainder__Sale Condition
1047,0.0,0.0,0.0,0.0,1.0,1048,527453160,160,RL,24.0,2308,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,NPkVill,Norm,Norm,2Story,6,6,1975,1975,Gable,CompShg,Plywood,Brk Cmn,NaN,0.0,TA,TA,CBlock,Gd,TA,No,ALQ,286.0,LwQ,294.0,275.0,855.0,GasA,Gd,Y,SBrkr,855,601,0,1456,0.0,0.0,2,1,4,1,TA,7,Typ,0,NaN,Attchd,1975.0,RFn,2.0,460.0,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,5,2008,WD,Normal
1975,1.0,0.0,0.0,0.0,0.0,1976,902106070,50,RM,50.0,6000,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,1.5Fin,6,6,1927,1950,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,BrkTil,TA,TA,No,Rec,276.0,Unf,0.0,569.0,845.0,GasA,TA,Y,SBrkr,866,430,0,1296,0.0,0.0,1,0,3,1,TA,6,Typ,0,NaN,Detchd,1980.0,Unf,2.0,576.0,TA,TA,Y,0,0,175,0,0,0,NaN,NaN,NaN,0,7,2007,WD,Normal
2601,1.0,0.0,0.0,0.0,0.0,2602,535380010,70,RL,60.0,7200,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,2Story,6,8,1915,1950,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,Ex,CBlock,TA,TA,No,Rec,338.0,Unf,0.0,325.0,663.0,GasA,Ex,Y,SBrkr,774,821,0,1595,0.0,0.0,2,0,3,1,TA,7,Typ,1,Gd,Detchd,1974.0,Unf,2.0,528.0,TA,TA,Y,49,0,231,0,0,0,NaN,NaN,NaN,0,4,2006,WD,Normal
1133,1.0,0.0,0.0,0.0,0.0,1134,531373060,60,RL,67.0,12774,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,SawyerW,Norm,Norm,2Story,7,5,2003,2004,Gable,CompShg,VinylSd,VinylSd,BrkFace,95.0,Gd,TA,PConc,Gd,TA,Av,Unf,0.0,Unf,0.0,835.0,835.0,GasA,Ex,Y,SBrkr,835,828,0,1663,0.0,0.0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2.0,478.0,TA,TA,Y,168,68,0,0,0,0,NaN,NaN,NaN,0,7,2008,WD,Normal
1845,0.0,0.0,0.0,1.0,0.0,1846,533221060,160,FV,24.0,2117,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,2Story,6,5,2000,2000,Gable,CompShg,MetalSd,MetalSd,BrkFace,216.0,Gd,TA,PConc,Gd,TA,No,GLQ,417.0,Unf,0.0,339.0,756.0,GasA,Ex,Y,SBrkr,769,804,0,1573,0.0,0.0,2,1,3,1,Gd,4,Typ,0,NaN,Detchd,2000.0,Unf,2.0,440.0,TA,TA,Y,0,32,0,0,0,0,NaN,NaN,NaN,0,9,2007,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [ ]:
ct_inter.fit_transform(X_train_dummified)

In [45]:
# Using only the size and number of rooms.

X = ames[["Gr Liv Area", "TotRms AbvGrd"]]
y = ames["SalePrice"]

lr_pipeline = Pipeline(
  [("linear_regression", LinearRegression())]
)

lr_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_pipeline_fitted = lr_pipeline.fit(X_train,y_train)

y_pred = lr_pipeline_fitted.predict(X_test)

mean_squared_error(y_test, y_pred)


3321101266.1827025

In [47]:
# Using size, number of rooms, and building type.

X = ames[["Gr Liv Area", "TotRms AbvGrd", "Bldg Type"]]
y = ames["SalePrice"]

ct = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)

lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
)

lr_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_pipeline_fitted = lr_pipeline.fit(X_train,y_train)

y_pred = lr_pipeline_fitted.predict(X_test)

mean_squared_error(y_test, y_pred)

2860708045.4993176

In [53]:
ct_dummies = ColumnTransformer(
  [("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
  ("standardize", StandardScaler(), ["Gr Liv Area"])],
  
  remainder = "passthrough"
).set_output(transform = "pandas")

ct_inter = ColumnTransformer(
  [
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__Gr Liv Area", "dummify__Bldg Type_1Fam"]), 
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__Gr Liv Area", "dummify__Bldg Type_2fmCon"]),
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__Gr Liv Area", "dummify__Bldg Type_Duplex"]),
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__Gr Liv Area", "dummify__Bldg Type_Twnhs"]),
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__Gr Liv Area", "dummify__Bldg Type_TwnhsE"])
  ],
  remainder = "drop"
).set_output(transform = "pandas")

X_train_dummified = ct_dummies.fit_transform(X_train)
X_train_dummified



,dummify__Bldg Type_1Fam,dummify__Bldg Type_2fmCon,dummify__Bldg Type_Duplex,dummify__Bldg Type_Twnhs,dummify__Bldg Type_TwnhsE,standardize__Gr Liv Area,remainder__TotRms AbvGrd
591,1.0,0.0,0.0,0.0,0.0,0.338948,7
1845,0.0,0.0,0.0,1.0,0.0,0.130635,4
86,1.0,0.0,0.0,0.0,0.0,-1.179592,5
1836,0.0,0.0,0.0,0.0,1.0,0.115061,5
2262,1.0,0.0,0.0,0.0,0.0,0.126742,7
...,...,...,...,...,...,...,...
1617,1.0,0.0,0.0,0.0,0.0,-0.069890,6
2751,1.0,0.0,0.0,0.0,0.0,-0.579963,6
2672,1.0,0.0,0.0,0.0,0.0,0.625134,9
1542,0.0,0.0,0.0,1.0,0.0,-0.922608,5


In [52]:
ct_inter.fit_transform(X_train_dummified)

TypeError: unhashable type: 'list'